# Inversion Experiments

In [ ]:
# For when connected to google colab
!git clone https://github.com/AdrSkapars/inversion_optimisation.git 
%cd inversion_optimisation
!uv pip install -e .

## Set Up Model

In [1]:
import sys
sys.path.insert(0, "/content/inversion_optimisation/src")

import torch
from datasets import load_dataset
from transformer_lens import HookedTransformer
import numpy as np
import random
import pickle
from tqdm import tqdm
import json
import logging
import time

from inversion_optimisation.utils import (
    get_paper_summary_stats_new, 
    DotDict,
    DATA_PATH,
)

/workspace/inversion_optimisation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the LLM model

device = ["cuda", "cpu"][0]
model_name = {
    # Simple models
    0: "attn-only-1l",
    1: "gelu-1l",
    2: "tiny-stories-1L-21M",
    
    ## Small models
    3: "tiny-stories-1M",
    4: "tiny-stories-3M",
    5: "tiny-stories-8M",
    6: "tiny-stories-28M",
    7: "tiny-stories-33M",
    8: "tiny-stories-instruct-33M",
    
    ## Large models
    9: "gpt2-small",
    10: "gpt2-medium",
    11: "gpt2-xl",
    12: "llama-7b",
    13: "meta-llama/Llama-3.2-1B",
    14: "meta-llama/Llama-3.2-1B-Instruct",
    15: "Qwen/Qwen2.5-0.5B",
    16: "Qwen/Qwen2.5-1.5B",
    17: "Qwen/Qwen2.5-3B",
    18: "pythia-1.4b",
}[7]

model = HookedTransformer.from_pretrained(model_name)#, device=device)
model = model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model tiny-stories-33M into HookedTransformer


# ------------------------

## Inv. Model Text-Only Inversion

In [ ]:
from inversion_optimisation.algorithms.text_inv_model import (
    LLMInversionModel, 
    train_inversion_model, 
    test_inversion_model_dataset, 
    test_inversion_model
)
from inversion_optimisation.inv_configs import INV_MODEL_TEXT_CFG

cfg = INV_MODEL_TEXT_CFG
# cfg = DotDict({
#     "t5_model_name" : "t5-small",
#     "t5_tokenizer_name" : "t5-small",
#     "llm_model_name" : "roneneldan/TinyStories-33M",
#     "num_generation_tokens" : 25,
#     "seed" : 24,
#     "dataset_size" : 400000,
#     "min_seq_length" : 1,
#     "max_seq_length" : 10,
#     "max_length" : 16,
#     "val_split" : 0.1,
#     # "output_dir" : "/content/TextInvModel_Saves",
#     "batch_size" : 160,
#     "mini_batch_size" : 160,
#     "num_epochs" : 30,
#     "save_steps" : 3000000,
#     "warmup_steps" : 1000,
#     "num_workers" : 12,
#     "learning_rate" : 1e-3,
#     "weight_decay" : 0.05,
#     # "dataset" : ["random", "reddit", "tinystories"][0],
# })

In [ ]:
# Load initial model
inv_model = LLMInversionModel(
    t5_model_name=cfg.t5_model_name,
    t5_tokenizer_name=cfg.t5_tokenizer_name,
    llm_model_name=cfg.llm_model_name,
    num_generation_tokens=cfg.num_generation_tokens,
).to(device)

In [ ]:
# Set level of logging
# logging.getLogger().setLevel(logging.INFO)
# logging.getLogger().setLevel(logging.WARNING)
logging.getLogger().setLevel(logging.ERROR)

In [ ]:
# Train model
elapsed_time = time.time()
inv_model = train_inversion_model(
    model=inv_model,
    seed=cfg.seed, # Set random seed
    dataset_size=cfg.dataset_size,
    min_seq_length=cfg.min_seq_length,
    max_seq_length=cfg.max_seq_length,
    max_length=cfg.max_length,
    val_split=cfg.val_split,
    output_dir="data/TextInvModel_Saves",
    batch_size=cfg.batch_size,
    mini_batch_size=cfg.mini_batch_size,
    num_epochs=cfg.num_epochs,
    save_steps=cfg.save_steps,
    warmup_steps=cfg.warmup_steps,
    num_workers=cfg.num_workers,
    learning_rate=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    dataset=["random", "reddit", "tinystories"][0],
)
elapsed_time = time.time() - elapsed_time
print(f"Elapsed time: {elapsed_time:.2f}")

In [ ]:
# Evaluate model
num_targets = 1000
eval_batch_size = 200
input_len = 1
max_length = 16

# Load best model
inv_model = LLMInversionModel.from_pretrained("data/TextInvModel_Saves/best_model").to(device)

# Generate dataset
torch.manual_seed(1)
np.random.seed(1)
random.seed(1)

# Choose random (Random) dataset
tokens_list = []
for _ in tqdm(range(num_targets)):
    tokens = torch.randint(0, len(inv_model.tokenizer.vocab), (1, input_len)).to(device)
    tokens_list.append(tokens)
loaded_true_tokens = torch.cat(tokens_list, dim=0).to(device)
loaded_text_samples = inv_model.llm_tokenizer.batch_decode(loaded_true_tokens)

# # Choose reddit (NL OOD) dataset
# loaded_true_tokens = load_dataset_tokens("reddit", input_len, num_targets, include_bos=False, random_sentence=True, random_start=False)
# loaded_text_samples = inv_model.llm_tokenizer.batch_decode(loaded_true_tokens)

# # Choose tinystories (NL ID) dataset
# loaded_true_tokens = load_dataset_tokens("tinystories", input_len, num_targets, include_bos=False, random_sentence=True, random_start=False)
# loaded_text_samples = inv_model.llm_tokenizer.batch_decode(loaded_true_tokens)

# Run model
results, stats = test_inversion_model_dataset(inv_model, loaded_text_samples, eval_batch_size, input_len, max_length)

# with open(DATA_PATH / f'TextInvModel_Saves/stats_{input_len}_{num_targets}.json', 'w') as f:
#     json.dump(stats, f)
# with open(DATA_PATH / f'TextInvModel_Saves/results_{input_len}_{num_targets}.pkl', 'wb') as f:
#     pickle.dump(results, f)

In [ ]:
# Example test model
test_texts = ["Hey","Hello","I","So", 'American']
results = test_inversion_model(model, test_texts, max_length=8)

## Reverse Model Text-Only Inversion

In [ ]:
from inversion_optimisation.algorithms.reverse_model import reverse_model_text_search
from inversion_optimisation.inv_configs import REVERSE_MODEL_TEXT_CFG

cfg = REVERSE_MODEL_TEXT_CFG

In [ ]:
# Set parameters for dataset size
cfg.input_len = 1
cfg.output_len = 25
cfg.num_targets = 1000
cfg.target_strategy = ["random", "reddit", "tinystories"][0]

# Set random seeds
torch.manual_seed(1)
np.random.seed(1)
random.seed(1)

# Generate the targets used for all experiments
tokens_list = []
for _ in tqdm(range(cfg.num_targets)):
    tokens = torch.randint(0, len(model.tokenizer.vocab), (1, cfg.input_len)).to(device)
    tokens_list.append(tokens)
true_tokens = torch.cat(tokens_list, dim=0).to(device)
with open(DATA_PATH / f'true_tokens_{cfg.num_targets}_{cfg.input_len}.pkl', 'wb') as file:
    pickle.dump(true_tokens, file)

batch_size = 1000
for batch in range(0, cfg.num_targets, batch_size):
    input_tokens = true_tokens[batch:batch+batch_size].to(device)
    output_tokens = model.generate(
        input_tokens,
        max_new_tokens=cfg.output_len,
        do_sample=False,
        stop_at_eos=False,
        verbose=False,
        return_type="tokens",)[:,cfg.input_len:]
    if batch == 0:
        all_output_tokens = output_tokens
    else:
        all_output_tokens = torch.cat((all_output_tokens, output_tokens), dim=0)
with open(DATA_PATH / f"true_tokens_{cfg.num_targets}_{cfg.input_len}_{cfg.output_len}_greedy.pkl", 'wb') as file:
    pickle.dump(all_output_tokens, file)

results, elapsed_time = reverse_model_text_search(cfg, model, device)
print()
stats = get_paper_summary_stats_new(results, epochs=1)
stats["elapsed_time"] = elapsed_time
stats["experiment_params"] = cfg
print("input_len", cfg.input_len, "\tpercent_zero_loss", stats["percent_zero_loss"], "\tpercent_exact_inversion", stats["percent_exact_inversion"], "\telapsed_time", stats["elapsed_time"])

# with open(DATA_PATH / f'{cfg.save_folder}/stats_{cfg.input_len}_{cfg.num_targets}.json', 'w') as f:
#     json.dump(stats, f)
# with open(DATA_PATH / f'{cfg.save_folder}/results_{cfg.input_len}_{cfg.num_targets}.pkl', 'wb') as f:
#     pickle.dump(results, f)